# Colab 11 (docente) — ¿Puedo medir la viscosidad de un fluido dejando caer una esfera?

**Laboratorio 1 · Departamento de Física · FCEN-UBA**

Clase 11 — 21/10

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/charlyacha/Labo1-colabs/blob/main/11_docente_Viscosidad_por_caida_de_esferas.ipynb)

> **Versión resuelta, para el docente.** Este cuaderno tiene el análisis completo del que
> el `11_estudiante` es el andamiaje. Sirve para consultar la solución mientras se circula
> por el aula y para tener a mano los números de referencia. **No se reparte:** la clase es
> el ensayo general de la Práctica Especial y el sentido es que el grupo lo arme solo.
>
> Los datos son sintéticos, con semilla fija, y están diseñados para exhibir los tres puntos
> de la clase: la corrección de pared **domina** en las esferas grandes, el ajuste ponderado
> con ordenada libre da una ordenada compatible con cero, y η recupera el valor de tabla.
> Reemplazá el bloque de datos por los reales.

### 1. La física

Una esfera que cae en un fluido viscoso alcanza una **velocidad límite** cuando el peso, el
empuje y el arrastre se equilibran. Con arrastre de Stokes ($F = 6\pi\eta r v$, válido a bajo
número de Reynolds):

$$v_{\lim} = \frac{2}{9}\,\frac{(\rho_e - \rho_f)\,g}{\eta}\,r^2$$

o sea, **$v_{\lim}$ contra $r^2$ es una recta por el origen**, y de su pendiente sale $\eta$.

Dos condiciones de validez que se **verifican, no se suponen**:

- **Número de Reynolds.** $\mathrm{Re} = \rho_f\,v\,(2r)/\eta$; Stokes vale para
  $\mathrm{Re} \ll 1$. El apartamiento aparece en los residuos.
- **Efecto de pared.** El tubo no es infinito. Ladenburg, a primer orden:
  $v_{\infty} \approx v_{\text{medida}}\,(1 + 2{,}104\,r/R_{\text{tubo}})$. Con $r/R \sim 0{,}2$
  ya es una corrección del 40 %.

In [ ]:
import os

if not os.path.exists("lab1_utils.py"):
    !wget -q -O lab1_utils.py https://raw.githubusercontent.com/charlyacha/Labo1-colabs/main/lab1_utils.py

import numpy as np
import matplotlib.pyplot as plt
import sympy as sp
import lab1_utils as lab

lab.estilo_lab1()
print("Listo. numpy", np.__version__)

### Celda 1 — Los datos

Esferas de acero en glicerina, cronometradas entre dos marcas del tubo.

In [ ]:
# ============ DATOS DE EJEMPLO - REEMPLAZAR POR LOS PROPIOS ============
rng = np.random.default_rng(11)

# --- constantes del montaje (con su incerteza) ---
g       = 9.81
rho_e, s_rho_e = 7800.0, 30.0     # acero, kg/m^3
rho_f, s_rho_f = 1260.0, 5.0      # glicerina, kg/m^3
R_tubo, s_Rtubo = 12.5e-3, 0.1e-3 # radio interno del tubo, m
L,      s_L     = 0.150, 0.001    # separacion entre marcas, m

# valores "verdaderos" que la simulacion usa para fabricar los datos
ETA_VERDADERO = 1.00              # Pa*s (glicerina ~22 C) - la respuesta a recuperar

# --- esferas: seis diametros, factor 5 en r (factor 25 en r^2) ---
diam_mm = np.array([1.0, 1.5, 2.0, 3.0, 4.0, 5.0])
r  = diam_mm/2 * 1e-3
s_r = np.full_like(r, 0.005e-3)   # micrometro, ~5 um

# velocidad limite infinita (Stokes) y velocidad MEDIDA (frenada por la pared)
v_inf_true = (2/9)*(rho_e - rho_f)*g*r**2/ETA_VERDADERO
v_meas_true = v_inf_true/(1 + 2.104*r/R_tubo)

# --- cronometrado: 5 pasadas por esfera, tiempo = L/v con ruido ---
n_rep = 5
t_true = L/v_meas_true
tiempos = t_true[:,None]*(1 + rng.normal(0, 0.012, size=(len(r), n_rep)))

# velocidad medida por esfera y su incerteza (SEM propagado a v = L/t)
t_media = tiempos.mean(axis=1)
s_t     = tiempos.std(axis=1, ddof=1)/np.sqrt(n_rep)
v_meas  = L/t_media
s_v     = v_meas*np.sqrt((s_t/t_media)**2 + (s_L/L)**2)

print(f"{'d (mm)':>7} {'v_medida (mm/s)':>18} {'sigma_v (mm/s)':>16}")
for d, vv, sv in zip(diam_mm, v_meas, s_v):
    print(f"{d:7.1f} {1e3*vv:18.3f} {1e3*sv:16.3f}")

### Celda 2 — Velocidad límite de cada esfera, con su incerteza

Ya salió de la Celda 1 (`v_meas`, `s_v`). Verificamos que la dispersión relativa es la que da el cronometrado.

In [ ]:
rel = 100*s_v/v_meas
for d, vv, sv, rr in zip(diam_mm, v_meas, s_v, rel):
    print(f"d = {d:.1f} mm:  v = {lab.formatear(1e3*vv, 1e3*sv, 'mm/s')}   ({rr:.1f} %)")

### Celda 3 — Corrección por efecto de pared

$v_\infty = v_{\text{medida}}\,(1 + 2{,}104\,r/R_{\text{tubo}})$. La corrección **también tiene
incerteza**, porque $r$ y $R_{\text{tubo}}$ la tienen. Se propaga.

In [ ]:
k = 2.104
factor = 1 + k*r/R_tubo
v_corr = v_meas*factor

# propagacion: v_corr = v_meas*(1 + k r/R). Fuentes: v_meas, r, R_tubo.
dvc_dv = factor
dvc_dr = v_meas*k/R_tubo
dvc_dR = -v_meas*k*r/R_tubo**2
s_vcorr = np.sqrt((dvc_dv*s_v)**2 + (dvc_dr*s_r)**2 + (dvc_dR*s_Rtubo)**2)

print(f"{'d (mm)':>7} {'r/R':>7} {'correccion':>11} {'v_corr (mm/s)':>15}")
for d, rc, fa, vc in zip(diam_mm, r/R_tubo, factor, v_corr):
    print(f"{d:7.1f} {rc:7.3f} {100*(fa-1):9.1f}% {1e3*vc:15.3f}")
print(f"\nLa esfera mas grande se corrige un {100*(factor[-1]-1):.0f} %: la pared NO es despreciable.")

### Celda 4 — Ajuste ponderado de $v$ contra $r^2$, con diagnóstico completo

Ordenada al origen **libre**: el modelo dice que es cero, así que si sale distinta de cero con
significancia, aprendimos algo (un sistemático en el cronometrado). Primero mostramos por qué
la corrección de pared no es opcional: ajustamos **sin** corregir y **con** corrección.

In [ ]:
def recta(r2, a, b):
    return a*r2 + b

r2 = r**2

print("=== SIN corregir la pared (incorrecto) ===")
p_mal, e_mal, _ = lab.ajustar(recta, r2, v_meas, yerr=s_v,
                              nombres=["pendiente", "ordenada"])

print("\n=== CON correccion de pared (correcto) ===")
p, e, cov = lab.ajustar(recta, r2, v_corr, yerr=s_vcorr,
                        nombres=["pendiente", "ordenada"])

lab.grafico_con_residuos(r2, v_corr, recta, p, yerr=s_vcorr,
                         xlabel=r"$r^2$  [m$^2$]", ylabel=r"$v_\infty$  [m/s]",
                         titulo="Velocidad limite corregida vs. r^2",
                         etiqueta_modelo="ajuste ponderado")
plt.show()

lab.reportar(1e3*p[1], 1e3*e[1], "mm/s", nombre="ordenada al origen")
print(f"  -> compatible con cero a {abs(p[1]/e[1]):.1f} sigma")

### Celda 5 — Despejá $\eta$ y propagá la incerteza

De la pendiente $m = \dfrac{2}{9}\dfrac{(\rho_e-\rho_f)g}{\eta}$ sale
$\eta = \dfrac{2}{9}\dfrac{(\rho_e-\rho_f)g}{m}$. Propagamos desde la pendiente y las dos
densidades con `lab.propagar` (SymPy).

In [ ]:
m_sym, re_sym, rf_sym = sp.symbols("m rho_e rho_f", positive=True)
g_sym = sp.symbols("g", positive=True)
eta_expr = sp.Rational(2,9)*(re_sym - rf_sym)*g_sym/m_sym

variables = [m_sym, re_sym, rf_sym]
valores = {m_sym: p[0], re_sym: rho_e, rf_sym: rho_f, g_sym: g}
errores = {m_sym: e[0], re_sym: s_rho_e, rf_sym: s_rho_f}

eta, s_eta = lab.propagar(eta_expr, variables, valores, errores, nombre="eta")
print()
lab.reportar(eta, s_eta, "Pa*s", nombre="eta medido")
print(f"eta de tabla (referencia): {ETA_VERDADERO:.3f} Pa*s")
lab.compatibilidad(eta, s_eta, ETA_VERDADERO, 0.0,
                   etiquetas=("eta medido", "eta tabla"))

### Celda 6 — Número de Reynolds punto por punto

$\mathrm{Re} = \rho_f\,v\,(2r)/\eta$. Vemos cuáles cumplen $\mathrm{Re}\ll 1$, rehacemos el
ajuste solo con ésos y comparamos $\eta$.

In [ ]:
Re = rho_f*v_corr*(2*r)/eta
print(f"{'d (mm)':>7} {'Re':>8}")
for d, re_ in zip(diam_mm, Re):
    marca = "  <- Re no es << 1" if re_ > 0.3 else ""
    print(f"{d:7.1f} {re_:8.3f}{marca}")

seguros = Re < 0.3
print(f"\nEsferas con Re < 0.3: {seguros.sum()} de {len(r)}")

if seguros.sum() >= 3 and seguros.sum() < len(r):
    p_s, e_s, _ = lab.ajustar(recta, r2[seguros], v_corr[seguros],
                              yerr=s_vcorr[seguros],
                              nombres=["pendiente", "ordenada"], verbose=False)
    eta_s, s_eta_s = lab.propagar(eta_expr, variables,
                                  {m_sym: p_s[0], re_sym: rho_e, rf_sym: rho_f, g_sym: g},
                                  {m_sym: e_s[0], re_sym: s_rho_e, rf_sym: s_rho_f},
                                  nombre="eta (Re<0.3)", verbose=False)
    print()
    lab.reportar(eta,   s_eta,   "Pa*s", nombre="eta (todas)   ")
    lab.reportar(eta_s, s_eta_s, "Pa*s", nombre="eta (Re<0.3)  ")
    lab.compatibilidad(eta, s_eta, eta_s, s_eta_s,
                       etiquetas=("todas", "Re<0.3"))
else:
    print("En este rango todas las esferas cumplen Re < 0.3: Stokes vale en todo el conjunto.")

### 4. Autoevaluación — respondida

1. **¿La ordenada es compatible con cero?** Sí (ver Celda 4): a menos de ~1 σ. No hay
   evidencia de un sistemático en el cronometrado del tipo "arranco tarde el cronómetro".
2. **¿χ²ᵥ cerca de 1?** Sí. Si diera mucho mayor, se distingue *barras* de *modelo* mirando
   los residuos: barras mal estimadas dan residuos grandes pero **sin estructura**; modelo
   mal especificado da residuos **con forma** (curvatura, escalón).
3. **¿Qué medición domina la incerteza de η?** Con estos datos, **ninguna sola**: la tabla de
   contribuciones de la Celda 5 reparte la varianza casi en partes iguales entre la pendiente
   (el cronometrado, ~48 %) y la densidad de las esferas $\rho_e$ (~51 %); $\rho_f$ es
   despreciable (~1 %). Ésa es la lección de la tabla, y suele sorprender: mejorar el
   cronometrado **y** pesar mejor las esferas ayudan por igual; medir mejor la densidad del
   fluido no sirve de nada. (Si en tus datos el cronometrado domina más claramente, la
   conclusión cambia — por eso la tabla se mira, no se supone.)
4. **¿η compatible con tabla?** Sí, **una vez corregida la pared**. Sin corregir, la pendiente
   es menor (las esferas grandes caen "de más" lento por la pared) y η sale
   sistemáticamente **alta**: es el contraste de la Celda 4.
5. **¿Cambia η al sacar las esferas grandes?** En este conjunto, dentro del error (Celda 6):
   Stokes se sostiene en todo el rango medido. Con esferas aún más grandes, o un fluido menos
   viscoso, el Re crecería y aparecería en los residuos.
6. **¿Se podría distinguir Stokes de un arrastre cuadrático?** Con este rango de Re, no:
   ambos modelos ajustan comparablemente. Decirlo es una conclusión legítima y superior a
   inventar una — es exactamente la lección de R² del Colab 10 trasladada acá.

### 5. Propuesta de Práctica Especial

Igual que en la versión del estudiante: para el **28/10**, media carilla con pregunta física,
magnitud a determinar y precisión esperada, modelo, variable independiente y rango, estimación
**a priori** del error del resultado, y equipamiento.